# Обучение ruBERT для классификации направления сигнала ЦБ

Этот notebook обучает direction model для проекта `telegram_signal_searcher`.

Главная постановка:

```text
Input:  [TOPIC=t_i] text
Output: direction = "+" или "-"
```

Модель не определяет тему Telegram-поста. Тема уже должна быть найдена первой topic model. Direction model отвечает на следующий вопрос: если пост относится к теме `t_i`, он усиливает сигнал ЦБ по этой теме или ослабляет его?

Основной режим notebook — `presplit`: используются уже подготовленные файлы `direction_train.parquet` и `direction_test.parquet`. Fallback-режим `raw_wide` оставлен для ситуации, когда нужно заново построить long dataset из исходной wide-разметки.

## 1. Постановка задачи

В проекте есть первая модель, которая решает multi-label topic classification:

```text
text -> t1_relevant, t2_relevant, t3_relevant, t4_relevant, t5_relevant
```

Direction model обучается как вторая стадия:

```text
[TOPIC=t_i] text -> "+" / "-"
```

`+` означает, что пост усиливает или подтверждает сигнал Центрального банка по данной теме. `-` означает, что пост ослабляет, отрицает или ставит под сомнение этот сигнал.

## 2. Связь с topic model

Direction model идёт после topic model, потому что направление без темы неоднозначно. Один и тот же пост может относиться сразу к нескольким темам, и направление для этих тем может отличаться.

Например:

```text
Topic model:
  text -> t1=1, t4=1

Direction model:
  [TOPIC=t1] text -> "+"
  [TOPIC=t4] text -> "-"
```

Поэтому обучающая строка здесь — это не просто пост, а пара `пост x тема`.

## 3. Формат входа `[TOPIC=t_i] text`

ruBERT получает один текстовый вход. Чтобы модель знала, для какой темы нужно определить направление, тема явно добавляется в начало:

```text
[TOPIC=t1] текст поста
[TOPIC=t4] тот же текст поста
```

Такой формат позволяет использовать одну бинарную модель для всех пяти тем. Модель учится учитывать и сам текст, и явный маркер темы.

## 4. Режимы входных данных: `presplit` и `raw_wide`

Notebook поддерживает два режима:

`presplit` — основной режим для текущего проекта. Это означает, что train/test уже разделены на уровне файлов:

```text
direction_train.parquet
direction_test.parquet
```

Важно: `presplit` не означает, что файлы уже находятся в long-формате. Реальные файлы проекта могут быть wide-таблицами с колонками `t1_relevant ... t5_relevant` и `t1_direction ... t5_direction`. Поэтому notebook сначала автоматически определяет схему, при необходимости выполняет `wide -> long`, а уже потом делает validation split из train-файла.

`raw_wide` — fallback-режим. Читается исходный файл:

```text
annotated_direct_revised.parquet
```

Затем wide-разметка преобразуется в long-формат `post_uid | text | model_text | topic | direction | label_id`.

## 5. Импорты и базовые настройки

CUDA-пакеты отдельно не устанавливаются. Kaggle уже даёт PyTorch GPU окружение. Если в конкретном Kaggle image не хватает `transformers` или `datasets`, можно один раз раскомментировать install-строку ниже.

In [ ]:
# !pip install -q transformers datasets accelerate scikit-learn

from pathlib import Path
import json
import os
import random
import shutil
import zipfile

import numpy as np
import pandas as pd
import torch
from IPython.display import FileLink, display

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
    roc_auc_score,
)

In [ ]:
IS_KAGGLE = True

DATA_INPUT_MODE = "presplit"  # "presplit" или "raw_wide"
USE_EXISTING_SPLIT_COLUMN = False
SMOKE_RUN = False

KAGGLE_TRAIN_PATH = Path("/kaggle/input/<dataset-name>/direction_train.parquet")
KAGGLE_TEST_PATH = Path("/kaggle/input/<dataset-name>/direction_test.parquet")
KAGGLE_RAW_WIDE_PATH = Path("/kaggle/input/<dataset-name>/annotated_direct_revised.parquet")

LOCAL_TRAIN_PATH = Path("../data/annotated_data/direction_train.parquet")
LOCAL_TEST_PATH = Path("../data/annotated_data/direction_test.parquet")
LOCAL_RAW_WIDE_PATH = Path("../data/annotated_data/annotated_direct_revised.parquet")

TRAIN_PATH = KAGGLE_TRAIN_PATH if IS_KAGGLE else LOCAL_TRAIN_PATH
TEST_PATH = KAGGLE_TEST_PATH if IS_KAGGLE else LOCAL_TEST_PATH
RAW_WIDE_DATA_PATH = KAGGLE_RAW_WIDE_PATH if IS_KAGGLE else LOCAL_RAW_WIDE_PATH

WORK_DIR = Path("/kaggle/working") if IS_KAGGLE else Path("../outputs/direction_training")
DATASET_DIR = WORK_DIR / "direction_dataset"
ERROR_DIR = WORK_DIR / "direction_errors"
REPORTS_PACKAGE_DIR = WORK_DIR / "direction_reports_package"
ALL_OUTPUTS_PACKAGE_DIR = WORK_DIR / "direction_all_outputs_package"

TOPICS = ["t1", "t2", "t3", "t4", "t5"]

RELEVANT_COLS = {
    "t1": "t1_relevant",
    "t2": "t2_relevant",
    "t3": "t3_relevant",
    "t4": "t4_relevant",
    "t5": "t5_relevant",
}

DIRECTION_COLS = {
    "t1": "t1_direction",
    "t2": "t2_direction",
    "t3": "t3_direction",
    "t4": "t4_direction",
    "t5": "t5_direction",
}

REQUIRED_LONG_COLUMNS = ["text", "topic", "direction", "label_id"]
META_COLS = [
    "orig_row",
    "channel_id",
    "channel_name",
    "channel_username",
    "message_id",
    "post_id",
    "post_date",
    "date",
    "source_file",
    "direction_reasoning",
]

LABEL2ID = {"-": 0, "+": 1}
ID2LABEL = {0: "-", 1: "+"}

MODEL_SIZE = "large"  # "base" или "large"
CLASS_WEIGHT_MODE = "balanced"  # "balanced", "sqrt", "none"
CLASS_WEIGHT_CAP = 10.0

MAX_LENGTH = 512
TEXT_COL = "model_text"

VALID_SIZE = 0.15
RAW_WIDE_TRAIN_SIZE = 0.70
RAW_WIDE_TEST_SIZE = 0.15
SEED = 42

EPOCHS = 4
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01

DATASET_DIR.mkdir(parents=True, exist_ok=True)
ERROR_DIR.mkdir(parents=True, exist_ok=True)

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

print("DATA_INPUT_MODE:", DATA_INPUT_MODE)
print("WORK_DIR:", WORK_DIR)

## 6. Загрузка train/test parquet

В режиме `presplit` notebook ищет `direction_train.parquet` и `direction_test.parquet`. Для Kaggle путь зависит от имени подключённого Dataset, поэтому если явный путь не найден, используется auto-search по `/kaggle/input`.

In [ ]:
def find_file(filename, root="/kaggle/input"):
    root_path = Path(root)
    if not root_path.exists():
        raise FileNotFoundError(f"Search root does not exist: {root_path}")

    matches = list(root_path.rglob(filename))
    if not matches:
        raise FileNotFoundError(f"Could not find {filename} under {root}")

    if len(matches) > 1:
        print(f"Found several candidates for {filename}:")
        for match in matches:
            print(" -", match)

    return matches[0]


def resolve_file(path, filename, is_kaggle=IS_KAGGLE):
    path = Path(path)
    if path.exists():
        return path

    if is_kaggle:
        print(f"File not found by configured path: {path}")
        print(f"Trying Kaggle auto-search for {filename} ...")
        return find_file(filename, root="/kaggle/input")

    raise FileNotFoundError(f"File not found: {path}")

In [ ]:
def normalize_direction(value):
    if pd.isna(value):
        return np.nan

    value = str(value).strip().lower()

    plus_values = {
        "+",
        "plus",
        "positive",
        "pos",
        "up",
        "вверх",
        "плюс",
        "1",
    }

    minus_values = {
        "-",
        "minus",
        "negative",
        "neg",
        "down",
        "вниз",
        "минус",
        "0",
    }

    if value in plus_values:
        return "+"

    if value in minus_values:
        return "-"

    return np.nan


def is_truthy_relevant(value):
    if pd.isna(value):
        return False

    if isinstance(value, (bool, np.bool_)):
        return bool(value)

    if isinstance(value, (int, float, np.integer, np.floating)):
        return int(value) == 1

    value = str(value).strip().lower()

    return value in {
        "1",
        "true",
        "yes",
        "y",
        "да",
        "+",
        "relevant",
    }


def build_post_uid(frame):
    frame = frame.copy()

    if "post_uid" not in frame.columns:
        if {"channel_id", "message_id"}.issubset(frame.columns):
            frame["post_uid"] = frame["channel_id"].astype(str) + "_" + frame["message_id"].astype(str)
        elif "orig_row" in frame.columns:
            frame["post_uid"] = frame["orig_row"].astype(str)
        else:
            frame["post_uid"] = np.arange(len(frame)).astype(str)

    frame["post_uid"] = frame["post_uid"].astype(str)
    return frame


def wide_to_long_direction(frame, name):
    frame = frame.copy()
    frame = build_post_uid(frame)

    if "text" not in frame.columns:
        raise ValueError(f"{name}: missing required column 'text'")

    frame["text"] = frame["text"].fillna("").astype(str).str.strip()
    frame = frame[frame["text"].str.len() > 0].copy()

    meta_cols = [
        "post_uid",
        "orig_row",
        "channel_id",
        "channel_name",
        "channel_username",
        "message_id",
        "post_id",
        "post_date",
        "date",
        "source_file",
        "direction_reasoning",
    ]
    meta_cols = [col for col in meta_cols if col in frame.columns]

    long_parts = []

    for topic in TOPICS:
        rel_col = f"{topic}_relevant"
        dir_col = f"{topic}_direction"

        if dir_col not in frame.columns:
            print(f"{name}: skip {topic}, missing {dir_col}")
            continue

        tmp = frame.copy()
        tmp["topic"] = topic
        tmp["direction"] = tmp[dir_col].apply(normalize_direction)
        tmp = tmp[tmp["direction"].isin(["+", "-"])].copy()

        if rel_col in tmp.columns:
            relevant_mask = tmp[rel_col].apply(is_truthy_relevant)
            if relevant_mask.sum() > 0:
                tmp = tmp[relevant_mask].copy()
            else:
                print(
                    f"{name}: {rel_col} produced 0 truthy rows; "
                    f"keeping rows with valid {dir_col} only"
                )

        if tmp.empty:
            continue

        tmp["label_id"] = tmp["direction"].map(LABEL2ID).astype(int)
        tmp[TEXT_COL] = "[TOPIC=" + tmp["topic"].astype(str) + "] " + tmp["text"].astype(str)

        keep_cols = meta_cols + [
            "text",
            TEXT_COL,
            "topic",
            "direction",
            "label_id",
        ]
        keep_cols = list(dict.fromkeys(keep_cols))
        long_parts.append(tmp[keep_cols])

    if not long_parts:
        raise ValueError(
            f"{name}: could not build long dataset. "
            f"Expected columns like t1_direction ... t5_direction."
        )

    long_frame = pd.concat(long_parts, ignore_index=True)
    long_frame = long_frame.reset_index(drop=True)

    print(f"{name}: detected WIDE format and converted to LONG")
    print(f"{name}: source rows={len(frame)}")
    print(f"{name}: long rows={len(long_frame)}")
    print(f"{name}: unique post_uid={long_frame['post_uid'].nunique()}")

    return long_frame


def wide_to_long(raw_df):
    return wide_to_long_direction(raw_df, "raw_df")

In [ ]:
if DATA_INPUT_MODE == "presplit":
    TRAIN_PATH = resolve_file(TRAIN_PATH, "direction_train.parquet")
    TEST_PATH = resolve_file(TEST_PATH, "direction_test.parquet")

    train_full_df = pd.read_parquet(TRAIN_PATH)
    test_df = pd.read_parquet(TEST_PATH)

    print("Resolved TRAIN_PATH:", TRAIN_PATH)
    print("Resolved TEST_PATH:", TEST_PATH)
    print("train_full_df shape:", train_full_df.shape)
    print("test_df shape:", test_df.shape)
    print("train columns:", train_full_df.columns.tolist())
    print("test columns:", test_df.columns.tolist())
    display(train_full_df.head())
    display(test_df.head())
    train_full_df.info()
    test_df.info()

elif DATA_INPUT_MODE == "raw_wide":
    RAW_WIDE_DATA_PATH = resolve_file(RAW_WIDE_DATA_PATH, "annotated_direct_revised.parquet")
    raw_df = pd.read_parquet(RAW_WIDE_DATA_PATH)

    print("Resolved RAW_WIDE_DATA_PATH:", RAW_WIDE_DATA_PATH)
    print("raw_df shape:", raw_df.shape)
    print("raw columns:", raw_df.columns.tolist())
    display(raw_df.head())
    raw_df.info()

    direction_df = wide_to_long(raw_df)
    print("direction_df shape after wide -> long:", direction_df.shape)
    display(direction_df.head())

else:
    raise ValueError("DATA_INPUT_MODE must be 'presplit' or 'raw_wide'")

## 7. Проверка и нормализация схемы

Эта проверка нужна, чтобы notebook одинаково работал с двумя типами входа:

1. long-таблицами, где уже есть `text`, `topic`, `direction`, `label_id`;
2. wide-таблицами, где есть `t1_relevant ... t5_relevant` и `t1_direction ... t5_direction`.

В текущем проекте `direction_train.parquet` и `direction_test.parquet` уже разделены на train/test, но сами файлы находятся в wide-формате. Поэтому `presplit` означает только готовый train/test split, а не готовый long-format.

Модель обучается не на исходном `text`, а на `model_text` формата:

```text
[TOPIC=t_i] text
```

In [ ]:
def prepare_long_schema(frame, name):
    frame = frame.copy()

    missing_columns = [column for column in REQUIRED_LONG_COLUMNS if column not in frame.columns]
    if missing_columns:
        raise ValueError(f"{name}: missing required long columns: {missing_columns}")

    before = len(frame)

    frame = build_post_uid(frame)
    frame["text"] = frame["text"].fillna("").astype(str).str.strip()
    frame["topic"] = frame["topic"].astype(str).str.strip()
    frame["direction"] = frame["direction"].apply(normalize_direction)
    frame["label_id"] = frame["direction"].map(LABEL2ID)

    if TEXT_COL not in frame.columns:
        frame[TEXT_COL] = "[TOPIC=" + frame["topic"].astype(str) + "] " + frame["text"].astype(str)
    else:
        frame[TEXT_COL] = frame[TEXT_COL].fillna("").astype(str).str.strip()
        missing_model_text = frame[TEXT_COL].str.len() == 0
        frame.loc[missing_model_text, TEXT_COL] = (
            "[TOPIC="
            + frame.loc[missing_model_text, "topic"].astype(str)
            + "] "
            + frame.loc[missing_model_text, "text"].astype(str)
        )

    frame = frame[
        frame["topic"].isin(TOPICS)
        & frame["direction"].isin(["+", "-"])
        & frame["label_id"].isin([0, 1])
        & (frame["text"].str.len() > 0)
        & (frame[TEXT_COL].str.len() > 0)
    ].copy()

    frame["label_id"] = frame["label_id"].astype(int)
    frame["post_uid"] = frame["post_uid"].astype(str)
    frame = frame.reset_index(drop=True)

    after = len(frame)
    print(f"{name}: detected LONG format")
    print(f"{name}: rows before={before}, after={after}, dropped={before - after}")
    print(f"{name}: unique post_uid={frame['post_uid'].nunique()}")

    if frame.empty:
        raise ValueError(f"{name}: no rows left after schema normalization")

    return frame


def prepare_direction_schema(frame, name):
    print(f"{name}: columns:")
    print(frame.columns.tolist())

    has_long_columns = {"text", "topic", "direction"}.issubset(frame.columns)
    has_wide_direction_columns = any(
        f"{topic}_direction" in frame.columns
        for topic in TOPICS
    )

    if has_long_columns:
        return prepare_long_schema(frame, name)

    if has_wide_direction_columns:
        long_frame = wide_to_long_direction(frame, name)
        return prepare_long_schema(long_frame, name)

    raise ValueError(
        f"{name}: unknown direction dataset format. "
        f"Expected long columns text/topic/direction or wide columns t1_direction ... t5_direction. "
        f"Available columns: {frame.columns.tolist()}"
    )


if DATA_INPUT_MODE == "presplit":
    train_full_df = prepare_direction_schema(train_full_df, "train_full_df")
    test_df = prepare_direction_schema(test_df, "test_df")
else:
    direction_df = prepare_direction_schema(direction_df, "direction_df")

После auto-detect и `wide -> long` проверяем, что появились обучающие колонки:

```text
post_uid
text
model_text
topic
direction
label_id
```

Эта диагностика должна выполняться до validation split, чтобы сразу увидеть, что presplit-wide файлы корректно преобразованы.

In [ ]:
print("train_full_df shape:", train_full_df.shape if "train_full_df" in globals() else None)
print("test_df shape:", test_df.shape if "test_df" in globals() else None)

if DATA_INPUT_MODE == "presplit":
    display(train_full_df.head())
    display(test_df.head())

    print("Train direction distribution:")
    display(train_full_df["direction"].value_counts())

    print("Test direction distribution:")
    display(test_df["direction"].value_counts())

    print("Train topic x direction:")
    display(pd.crosstab(train_full_df["topic"], train_full_df["direction"]))

    print("Test topic x direction:")
    display(pd.crosstab(test_df["topic"], test_df["direction"]))
else:
    display(direction_df.head())
    print("Direction distribution:")
    display(direction_df["direction"].value_counts())
    print("Topic x direction:")
    display(pd.crosstab(direction_df["topic"], direction_df["direction"]))

## 8. Validation split без data leakage

Даже в long-формате split нужно делать по `post_uid`. Один пост может иметь несколько строк, если он относится к нескольким темам. Поэтому нельзя случайно разделять строки: одинаковый текст не должен попасть одновременно в train и validation.

В режиме `presplit` test уже задан файлом `direction_test.parquet`, а validation создаётся из `direction_train.parquet`. Если в train-файле есть колонка `split`, можно включить `USE_EXISTING_SPLIT_COLUMN = True`, но по умолчанию split строится заново по `post_uid`.

In [ ]:
def split_by_post_uid(frame, valid_size=VALID_SIZE, seed=SEED):
    unique_post_uids = frame["post_uid"].drop_duplicates()
    if len(unique_post_uids) < 2:
        raise ValueError("Need at least 2 unique post_uid values for train/valid split")

    train_uids, valid_uids = train_test_split(
        unique_post_uids,
        test_size=valid_size,
        random_state=seed,
        shuffle=True,
    )

    train_uid_set = set(train_uids.astype(str))
    valid_uid_set = set(valid_uids.astype(str))

    assert train_uid_set.isdisjoint(valid_uid_set)

    train_part = frame[frame["post_uid"].isin(train_uid_set)].reset_index(drop=True)
    valid_part = frame[frame["post_uid"].isin(valid_uid_set)].reset_index(drop=True)
    return train_part, valid_part


def split_raw_wide_long(frame):
    unique_post_uids = frame["post_uid"].drop_duplicates()
    if len(unique_post_uids) < 3:
        raise ValueError("Need at least 3 unique post_uid values for raw_wide train/valid/test split")

    train_uids, temp_uids = train_test_split(
        unique_post_uids,
        train_size=RAW_WIDE_TRAIN_SIZE,
        random_state=SEED,
        shuffle=True,
    )

    raw_valid_relative_size = VALID_SIZE / (VALID_SIZE + RAW_WIDE_TEST_SIZE)
    valid_uids, test_uids = train_test_split(
        temp_uids,
        train_size=raw_valid_relative_size,
        random_state=SEED,
        shuffle=True,
    )

    train_uid_set = set(train_uids.astype(str))
    valid_uid_set = set(valid_uids.astype(str))
    test_uid_set = set(test_uids.astype(str))

    assert train_uid_set.isdisjoint(valid_uid_set)
    assert train_uid_set.isdisjoint(test_uid_set)
    assert valid_uid_set.isdisjoint(test_uid_set)

    train_part = frame[frame["post_uid"].isin(train_uid_set)].reset_index(drop=True)
    valid_part = frame[frame["post_uid"].isin(valid_uid_set)].reset_index(drop=True)
    test_part = frame[frame["post_uid"].isin(test_uid_set)].reset_index(drop=True)
    return train_part, valid_part, test_part


if DATA_INPUT_MODE == "presplit":
    if USE_EXISTING_SPLIT_COLUMN and "split" in train_full_df.columns:
        split_values = train_full_df["split"].astype(str).str.lower().str.strip()
        train_df = train_full_df[split_values == "train"].reset_index(drop=True)
        valid_df = train_full_df[split_values.isin(["valid", "validation", "val"])].reset_index(drop=True)
        if train_df.empty or valid_df.empty:
            raise ValueError("Existing split column did not produce non-empty train and valid splits")
    else:
        train_df, valid_df = split_by_post_uid(train_full_df, valid_size=VALID_SIZE, seed=SEED)

    train_test_overlap = set(train_df["post_uid"]).intersection(set(test_df["post_uid"]))
    valid_test_overlap = set(valid_df["post_uid"]).intersection(set(test_df["post_uid"]))
    if train_test_overlap or valid_test_overlap:
        print("Warning: post_uid overlap with test split detected")
        print("train/test overlap:", len(train_test_overlap))
        print("valid/test overlap:", len(valid_test_overlap))

else:
    train_df, valid_df, test_df = split_raw_wide_long(direction_df)

if SMOKE_RUN:
    EPOCHS = 1
    train_df = train_df.sample(min(len(train_df), 200), random_state=SEED).reset_index(drop=True)
    valid_df = valid_df.sample(min(len(valid_df), 100), random_state=SEED).reset_index(drop=True)
    test_df = test_df.sample(min(len(test_df), 100), random_state=SEED).reset_index(drop=True)

print("train_df shape:", train_df.shape)
print("valid_df shape:", valid_df.shape)
print("test_df shape:", test_df.shape)
print("train post_uid:", train_df["post_uid"].nunique())
print("valid post_uid:", valid_df["post_uid"].nunique())
print("test post_uid:", test_df["post_uid"].nunique())

## 9. Сохранение подготовленных split-файлов

После нормализации и validation split сохраняем подготовленные версии датасетов. Эти файлы полезны для воспроизводимости: по ним видно, какие именно строки ушли в train, validation и test.

In [ ]:
prepared_splits = {
    "train": train_df,
    "valid": valid_df,
    "test": test_df,
}

for split_name, split_df in prepared_splits.items():
    parquet_path = DATASET_DIR / f"direction_{split_name}_prepared.parquet"
    csv_path = DATASET_DIR / f"direction_{split_name}_prepared.csv"
    split_df.to_parquet(parquet_path, index=False)
    split_df.to_csv(csv_path, index=False, encoding="utf-8-sig")
    print("Saved:", parquet_path)
    print("Saved:", csv_path)

## 10. Диагностика распределений

Для direction-задачи важно смотреть не только общий баланс классов `+` и `-`, но и распределение внутри каждой темы. Если, например, в `t4` мало плюсов или минусов, метрика по этой теме может быть нестабильной.

In [ ]:
def label_distribution_for_split(split_df, split_name):
    result = (
        split_df["direction"]
        .value_counts()
        .reindex(["-", "+"], fill_value=0)
        .rename_axis("direction")
        .reset_index(name="count")
    )
    result.insert(0, "split", split_name)
    return result


def topic_distribution(split_df):
    return pd.crosstab(split_df["topic"], split_df["direction"]).reindex(
        index=TOPICS,
        columns=["-", "+"],
        fill_value=0,
    )


label_distribution_by_split = pd.concat(
    [
        label_distribution_for_split(train_df, "train"),
        label_distribution_for_split(valid_df, "valid"),
        label_distribution_for_split(test_df, "test"),
    ],
    ignore_index=True,
)

train_topic_distribution = topic_distribution(train_df)
valid_topic_distribution = topic_distribution(valid_df)
test_topic_distribution = topic_distribution(test_df)

display(label_distribution_by_split)
display(train_topic_distribution)
display(valid_topic_distribution)
display(test_topic_distribution)

label_distribution_by_split.to_csv(
    WORK_DIR / "direction_label_distribution_by_split.csv",
    index=False,
    encoding="utf-8-sig",
)
train_topic_distribution.to_csv(WORK_DIR / "direction_topic_distribution_train.csv", encoding="utf-8-sig")
valid_topic_distribution.to_csv(WORK_DIR / "direction_topic_distribution_valid.csv", encoding="utf-8-sig")
test_topic_distribution.to_csv(WORK_DIR / "direction_topic_distribution_test.csv", encoding="utf-8-sig")

## 11. Архив подготовленных данных

Kaggle не всегда разрешает автоматическое скачивание файлов на компьютер из кода notebook. Поэтому notebook создаёт ZIP-архивы и показывает ссылки `FileLink`. Эти архивы также остаются в `/kaggle/working` и доступны во вкладке Output после завершения запуска.

In [ ]:
def make_zip_and_link(source_path, zip_name):
    source_path = Path(source_path)
    zip_base = WORK_DIR / zip_name

    if zip_base.with_suffix(".zip").exists():
        zip_base.with_suffix(".zip").unlink()

    archive_path = shutil.make_archive(
        base_name=str(zip_base),
        format="zip",
        root_dir=str(source_path),
    )

    print("Created archive:", archive_path)
    display(FileLink(archive_path))
    return archive_path


def zip_files_and_link(files, zip_filename):
    zip_path = WORK_DIR / zip_filename

    if zip_path.exists():
        zip_path.unlink()

    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for file in files:
            file = Path(file)
            if file.exists():
                zf.write(file, arcname=file.name)
            else:
                print("Skip missing file:", file)

    print("Created archive:", zip_path)
    display(FileLink(str(zip_path)))
    return zip_path


prepared_data_zip = make_zip_and_link(
    DATASET_DIR,
    "direction_prepared_data",
)

## 12. Выбор ruBERT base / large

`base` — быстрый baseline и проверка пайплайна.

`large` — основной эксперимент для итогового качества. Для `large` нужен GPU, `fp16`, маленький batch size и gradient checkpointing. TPU не используется: notebook рассчитан на PyTorch GPU в Kaggle.

In [ ]:
if MODEL_SIZE == "base":
    MODEL_NAME = "ai-forever/ruBert-base"
elif MODEL_SIZE == "large":
    MODEL_NAME = "ai-forever/ruBert-large"
else:
    raise ValueError("MODEL_SIZE must be 'base' or 'large'")

if MODEL_SIZE == "large":
    PER_DEVICE_TRAIN_BATCH_SIZE = 1
    PER_DEVICE_EVAL_BATCH_SIZE = 4
    GRADIENT_ACCUMULATION_STEPS = 16
    FP16 = True
    GRADIENT_CHECKPOINTING = True
else:
    PER_DEVICE_TRAIN_BATCH_SIZE = 8
    PER_DEVICE_EVAL_BATCH_SIZE = 16
    GRADIENT_ACCUMULATION_STEPS = 2
    FP16 = torch.cuda.is_available()
    GRADIENT_CHECKPOINTING = False

print("MODEL_NAME:", MODEL_NAME)
print("EPOCHS:", EPOCHS)
print("MAX_LENGTH:", MAX_LENGTH)
print("PER_DEVICE_TRAIN_BATCH_SIZE:", PER_DEVICE_TRAIN_BATCH_SIZE)
print("PER_DEVICE_EVAL_BATCH_SIZE:", PER_DEVICE_EVAL_BATCH_SIZE)
print("GRADIENT_ACCUMULATION_STEPS:", GRADIENT_ACCUMULATION_STEPS)
print("FP16:", FP16)
print("GRADIENT_CHECKPOINTING:", GRADIENT_CHECKPOINTING)

## 13. CUDA diagnostics

Эта ячейка проверяет, видит ли PyTorch GPU. Она не устанавливает CUDA-пакеты и не меняет окружение.

In [ ]:
print("torch:", torch.__version__)
print("cuda version:", torch.version.cuda)
print("cuda available:", torch.cuda.is_available())
print("device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

if torch.cuda.is_available():
    x = torch.tensor([1.0, 2.0, 3.0], device="cuda")
    y = x * 2
    print("CUDA smoke test:", y)

## 14. Токенизация и dynamic padding

Dynamic padding особенно важен для ruBERT-large: batch дополняется только до максимальной длины внутри batch, а не все тексты заранее до 512 токенов. Это снижает расход GPU-памяти.

In [ ]:
from transformers import AutoTokenizer, DataCollatorWithPadding

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def tokenize_batch(batch):
    return tokenizer(
        batch[TEXT_COL],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
    )


data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    pad_to_multiple_of=8,
)

## 15. Hugging Face Dataset

В отличие от multi-label topic model, здесь используется softmax и `CrossEntropyLoss`. Для конкретной пары `пост x тема` направление ровно одно: `+` или `-`.

Поэтому label хранится как `int64`:

```text
0 = "-"
1 = "+"
```

In [ ]:
from datasets import Dataset


def make_hf_dataset(split_df):
    hf_df = split_df[[TEXT_COL, "label_id"]].rename(columns={"label_id": "labels"}).copy()
    hf_df["labels"] = hf_df["labels"].astype(int)
    return Dataset.from_pandas(hf_df, preserve_index=False)


train_ds = make_hf_dataset(train_df)
valid_ds = make_hf_dataset(valid_df)
test_ds = make_hf_dataset(test_df)

train_ds = train_ds.map(tokenize_batch, batched=True)
valid_ds = valid_ds.map(tokenize_batch, batched=True)
test_ds = test_ds.map(tokenize_batch, batched=True)

train_ds = train_ds.remove_columns([TEXT_COL])
valid_ds = valid_ds.remove_columns([TEXT_COL])
test_ds = test_ds.remove_columns([TEXT_COL])

train_ds.set_format("torch")
valid_ds.set_format("torch")
test_ds.set_format("torch")

sample = train_ds[0]
print(sample.keys())
print(sample["labels"])
print(sample["labels"].dtype)

assert sample["labels"].dtype == torch.int64

## 16. Class weights

Class weights компенсируют дисбаланс между `+` и `-`. По умолчанию используется `balanced`, но можно выбрать `sqrt` для более мягкого взвешивания или `none`, чтобы отключить веса.

In [ ]:
class_counts = train_df["label_id"].value_counts().sort_index()
n_classes = 2
n_samples = len(train_df)

raw_weights = n_samples / (n_classes * class_counts)
raw_weights = raw_weights.reindex([0, 1]).fillna(1.0)

if CLASS_WEIGHT_MODE == "balanced":
    used_weights = raw_weights
elif CLASS_WEIGHT_MODE == "sqrt":
    used_weights = np.sqrt(raw_weights)
elif CLASS_WEIGHT_MODE == "none":
    used_weights = pd.Series([1.0, 1.0], index=[0, 1])
else:
    raise ValueError("Unknown CLASS_WEIGHT_MODE")

used_weights = used_weights.clip(upper=CLASS_WEIGHT_CAP)
class_weights = torch.tensor(used_weights.values, dtype=torch.float32)

class_weights_df = pd.DataFrame(
    {
        "label_id": [0, 1],
        "label": [ID2LABEL[0], ID2LABEL[1]],
        "class_count": class_counts.reindex([0, 1]).fillna(0).astype(int).values,
        "raw_weight": raw_weights.values,
        "used_weight": used_weights.values,
    }
)

display(class_weights_df)
class_weights_df.to_csv(WORK_DIR / "direction_class_weights.csv", index=False, encoding="utf-8-sig")
print("class_weights:", class_weights)

## 17. WeightedDirectionTrainer

Здесь используется `CrossEntropyLoss`, потому что задача binary single-label: для одной пары `пост x тема` есть ровно один правильный класс. `BCEWithLogitsLoss` и sigmoid нужны для multi-label задач, но не для этой постановки.

`compute_loss` принимает `**kwargs`, чтобы код не ломался на новых версиях `transformers`.

In [ ]:
from transformers import Trainer


class WeightedDirectionTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels").long()
        outputs = model(**inputs)
        logits = outputs.logits

        loss_fct = torch.nn.CrossEntropyLoss(
            weight=self.class_weights.to(logits.device) if self.class_weights is not None else None
        )

        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

## 18. Метрики

Macro F1 важнее accuracy, потому что классы могут быть несбалансированы. Нам нужно, чтобы модель хорошо отличала оба направления, а не просто угадывала более частый класс.

Дополнительно считаются ROC-AUC и PR-AUC для вероятности класса `+`.

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    probs = torch.softmax(torch.tensor(logits), dim=1).numpy()
    preds = np.argmax(probs, axis=1)

    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )
    precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(
        labels, preds, average="weighted", zero_division=0
    )

    metrics = {
        "accuracy": accuracy_score(labels, preds),
        "macro_precision": precision_macro,
        "macro_recall": recall_macro,
        "macro_f1": f1_macro,
        "weighted_precision": precision_weighted,
        "weighted_recall": recall_weighted,
        "weighted_f1": f1_weighted,
        "f1_minus": f1_score(labels, preds, pos_label=0, zero_division=0),
        "f1_plus": f1_score(labels, preds, pos_label=1, zero_division=0),
    }

    try:
        metrics["roc_auc"] = roc_auc_score(labels, probs[:, 1])
    except ValueError:
        metrics["roc_auc"] = 0.0

    try:
        metrics["pr_auc_plus"] = average_precision_score(labels, probs[:, 1])
    except ValueError:
        metrics["pr_auc_plus"] = 0.0

    return metrics

## 19. Загрузка модели и TrainingArguments

Для ruBERT-large включается gradient checkpointing: он снижает расход GPU-памяти и помогает обучать large-модель на Kaggle GPU. Цена — обучение становится немного медленнее.

Helper `make_training_args` сохраняет совместимость с версиями `transformers`, где параметр назывался `evaluation_strategy`, а не `eval_strategy`.

In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)

if GRADIENT_CHECKPOINTING:
    model.gradient_checkpointing_enable()
    model.config.use_cache = False


def make_training_args(**kwargs):
    try:
        return TrainingArguments(**kwargs)
    except TypeError as error:
        if "eval_strategy" in kwargs:
            kwargs = dict(kwargs)
            kwargs["evaluation_strategy"] = kwargs.pop("eval_strategy")
            return TrainingArguments(**kwargs)
        raise error


training_args = make_training_args(
    output_dir=str(WORK_DIR / f"rubert_{MODEL_SIZE}_direction_binary"),
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    num_train_epochs=EPOCHS,
    weight_decay=WEIGHT_DECAY,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    logging_steps=20,
    save_total_limit=2,
    report_to="none",
    fp16=FP16,
    gradient_checkpointing=GRADIENT_CHECKPOINTING,
    optim="adamw_torch",
    seed=SEED,
    data_seed=SEED,
)

training_args

## 20. Обучение

Следующая ячейка создаёт `Trainer`, а затем отдельная ячейка запускает обучение. Локально запускать обучение не нужно: эта часть предназначена для Kaggle GPU.

In [ ]:
trainer = WeightedDirectionTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    class_weights=class_weights,
)

Эта ячейка запускает обучение. Выполнять её стоит только на Kaggle GPU.

In [ ]:
train_result = trainer.train()

## 21. Оценка на test

После обучения оцениваем лучшую модель на `test_ds` и сохраняем построчные предсказания, вероятности классов и основные отчёты.

In [ ]:
test_output = trainer.predict(test_ds)

test_logits = test_output.predictions
test_probs = torch.softmax(torch.tensor(test_logits), dim=1).numpy()

test_pred_ids = np.argmax(test_probs, axis=1)
test_true_ids = test_df["label_id"].values

test_df_eval = test_df.copy()
test_df_eval["prob_minus"] = test_probs[:, 0]
test_df_eval["prob_plus"] = test_probs[:, 1]
test_df_eval["pred_label_id"] = test_pred_ids
test_df_eval["pred_direction"] = test_df_eval["pred_label_id"].map(ID2LABEL)
test_df_eval["is_correct"] = test_df_eval["label_id"] == test_df_eval["pred_label_id"]
test_df_eval["confidence"] = np.max(test_probs, axis=1)

precision_macro, recall_macro, _, _ = precision_recall_fscore_support(
    test_true_ids, test_pred_ids, average="macro", zero_division=0
)
precision_weighted, recall_weighted, _, _ = precision_recall_fscore_support(
    test_true_ids, test_pred_ids, average="weighted", zero_division=0
)

test_metrics = {
    "accuracy": float(accuracy_score(test_true_ids, test_pred_ids)),
    "macro_precision": float(precision_macro),
    "macro_recall": float(recall_macro),
    "macro_f1": float(f1_score(test_true_ids, test_pred_ids, average="macro", zero_division=0)),
    "weighted_precision": float(precision_weighted),
    "weighted_recall": float(recall_weighted),
    "weighted_f1": float(f1_score(test_true_ids, test_pred_ids, average="weighted", zero_division=0)),
    "f1_minus": float(f1_score(test_true_ids, test_pred_ids, pos_label=0, zero_division=0)),
    "f1_plus": float(f1_score(test_true_ids, test_pred_ids, pos_label=1, zero_division=0)),
}

try:
    test_metrics["roc_auc"] = float(roc_auc_score(test_true_ids, test_probs[:, 1]))
except ValueError:
    test_metrics["roc_auc"] = 0.0

try:
    test_metrics["pr_auc_plus"] = float(average_precision_score(test_true_ids, test_probs[:, 1]))
except ValueError:
    test_metrics["pr_auc_plus"] = 0.0

report_dict = classification_report(
    test_true_ids,
    test_pred_ids,
    target_names=["-", "+"],
    output_dict=True,
    zero_division=0,
)
report_df = pd.DataFrame(report_dict).T

cm = confusion_matrix(test_true_ids, test_pred_ids, labels=[0, 1])
cm_df = pd.DataFrame(cm, index=["true_-", "true_+"], columns=["pred_-", "pred_+"])

prediction_cols = [
    "post_uid",
    *META_COLS,
    "text",
    "model_text",
    "topic",
    "direction",
    "label_id",
    "prob_minus",
    "prob_plus",
    "pred_label_id",
    "pred_direction",
    "is_correct",
    "confidence",
]
prediction_cols = [column for column in prediction_cols if column in test_df_eval.columns]
prediction_cols = list(dict.fromkeys(prediction_cols))

test_df_eval[prediction_cols].to_csv(WORK_DIR / "direction_test_predictions.csv", index=False, encoding="utf-8-sig")
report_df.to_csv(WORK_DIR / "direction_classification_report.csv", encoding="utf-8-sig")
cm_df.to_csv(WORK_DIR / "direction_confusion_matrix.csv", encoding="utf-8-sig")

with open(WORK_DIR / "direction_metrics.json", "w", encoding="utf-8") as file:
    json.dump(test_metrics, file, ensure_ascii=False, indent=2)

display(pd.DataFrame([test_metrics]))
display(report_df)
display(cm_df)

## 22. Метрики по темам

Общая метрика может скрывать проблемы на отдельных темах. Поэтому отдельно смотрим качество по `t1`-`t5`.

In [ ]:
def safe_binary_count(values, label_id):
    return int((np.asarray(values) == label_id).sum())


topic_metric_rows = []

for topic in TOPICS:
    topic_part = test_df_eval[test_df_eval["topic"] == topic].copy()

    if topic_part.empty:
        topic_metric_rows.append(
            {
                "topic": topic,
                "n": 0,
                "accuracy": 0.0,
                "macro_f1": 0.0,
                "weighted_f1": 0.0,
                "f1_minus": 0.0,
                "f1_plus": 0.0,
                "true_plus": 0,
                "true_minus": 0,
                "pred_plus": 0,
                "pred_minus": 0,
            }
        )
        continue

    y_true = topic_part["label_id"].values
    y_pred = topic_part["pred_label_id"].values

    topic_metric_rows.append(
        {
            "topic": topic,
            "n": int(len(topic_part)),
            "accuracy": float(accuracy_score(y_true, y_pred)),
            "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
            "weighted_f1": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
            "f1_minus": float(f1_score(y_true, y_pred, pos_label=0, zero_division=0)),
            "f1_plus": float(f1_score(y_true, y_pred, pos_label=1, zero_division=0)),
            "true_plus": safe_binary_count(y_true, 1),
            "true_minus": safe_binary_count(y_true, 0),
            "pred_plus": safe_binary_count(y_pred, 1),
            "pred_minus": safe_binary_count(y_pred, 0),
        }
    )

topic_metrics_df = pd.DataFrame(topic_metric_rows)
topic_metrics_df.to_csv(WORK_DIR / "direction_topic_metrics.csv", index=False, encoding="utf-8-sig")

display(topic_metrics_df)

## 23. Анализ ошибок

Error analysis нужен для ручной проверки, какие именно посты модель путает: усиливающие сигнал как ослабляющие или наоборот. Отдельные файлы по темам позволяют понять, какие сигналы ЦБ сложнее всего интерпретируются Telegram-каналами.

In [ ]:
errors_df = test_df_eval[test_df_eval["is_correct"] == False].copy()

errors_df["error_type"] = np.where(
    (errors_df["direction"] == "+") & (errors_df["pred_direction"] == "-"),
    "plus_predicted_as_minus",
    "minus_predicted_as_plus",
)

if not errors_df.empty:
    errors_df = errors_df.sort_values(["topic", "error_type", "confidence"], ascending=[True, True, False])

errors_df.to_csv(ERROR_DIR / "direction_errors_all.csv", index=False, encoding="utf-8-sig")

if errors_df.empty:
    error_summary = pd.DataFrame(columns=["topic", "error_type", "n", "mean_confidence", "max_confidence"])
else:
    error_summary = (
        errors_df.groupby(["topic", "error_type"])
        .agg(
            n=("post_uid", "size"),
            mean_confidence=("confidence", "mean"),
            max_confidence=("confidence", "max"),
        )
        .reset_index()
        .sort_values(["topic", "error_type"])
    )

error_summary.to_csv(ERROR_DIR / "direction_error_summary.csv", index=False, encoding="utf-8-sig")

errors_df[errors_df["error_type"] == "plus_predicted_as_minus"].to_csv(
    ERROR_DIR / "plus_predicted_as_minus.csv", index=False, encoding="utf-8-sig"
)
errors_df[errors_df["error_type"] == "minus_predicted_as_plus"].to_csv(
    ERROR_DIR / "minus_predicted_as_plus.csv", index=False, encoding="utf-8-sig"
)

for topic in TOPICS:
    topic_errors = errors_df[errors_df["topic"] == topic].sort_values("confidence", ascending=False)
    topic_errors.to_csv(ERROR_DIR / f"{topic}_direction_errors.csv", index=False, encoding="utf-8-sig")

display(errors_df.head(30))
display(error_summary)

for topic in TOPICS:
    print(f"Top confident errors for {topic}")
    display(errors_df[errors_df["topic"] == topic].sort_values("confidence", ascending=False).head(5))

## 24. Архив отчётов

После test evaluation и error analysis создаётся архив `direction_reports.zip`. В него попадают метрики, предсказания, матрица ошибок, распределения и папка `direction_errors`.

In [ ]:
def reset_dir(path):
    path = Path(path)
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)
    return path


def copy_path_if_exists(source_path, target_dir):
    source_path = Path(source_path)
    target_dir = Path(target_dir)

    if not source_path.exists():
        print("Skip missing path:", source_path)
        return

    target_path = target_dir / source_path.name
    if source_path.is_dir():
        if target_path.exists():
            shutil.rmtree(target_path)
        shutil.copytree(source_path, target_path)
    else:
        shutil.copy2(source_path, target_path)


reports_files = [
    WORK_DIR / "direction_test_predictions.csv",
    WORK_DIR / "direction_metrics.json",
    WORK_DIR / "direction_classification_report.csv",
    WORK_DIR / "direction_confusion_matrix.csv",
    WORK_DIR / "direction_topic_metrics.csv",
    WORK_DIR / "direction_label_distribution_by_split.csv",
    WORK_DIR / "direction_class_weights.csv",
    WORK_DIR / "direction_topic_distribution_train.csv",
    WORK_DIR / "direction_topic_distribution_valid.csv",
    WORK_DIR / "direction_topic_distribution_test.csv",
]

reset_dir(REPORTS_PACKAGE_DIR)
for file in reports_files:
    copy_path_if_exists(file, REPORTS_PACKAGE_DIR)
copy_path_if_exists(ERROR_DIR, REPORTS_PACKAGE_DIR)

reports_zip = make_zip_and_link(REPORTS_PACKAGE_DIR, "direction_reports")

## 25. Сохранение модели

Финальная модель сохраняется в папку:

```text
/kaggle/working/rubert_<MODEL_SIZE>_direction_binary_<CLASS_WEIGHT_MODE>_final/
```

Дополнительно сохраняется `direction_model_config.json`, чтобы после обучения было понятно, какой режим данных, mapping классов и параметры эксперимента использовались.

In [ ]:
FINAL_MODEL_DIR = WORK_DIR / f"rubert_{MODEL_SIZE}_direction_binary_{CLASS_WEIGHT_MODE}_final"
FINAL_MODEL_DIR.mkdir(parents=True, exist_ok=True)

trainer.save_model(str(FINAL_MODEL_DIR))
tokenizer.save_pretrained(str(FINAL_MODEL_DIR))

direction_model_config = {
    "model_name": MODEL_NAME,
    "model_size": MODEL_SIZE,
    "task": "binary_topic_direction_classification",
    "input_format": "[TOPIC=t_i] text",
    "data_input_mode": DATA_INPUT_MODE,
    "train_path": str(TRAIN_PATH),
    "test_path": str(TEST_PATH),
    "raw_wide_data_path": str(RAW_WIDE_DATA_PATH),
    "text_col": TEXT_COL,
    "source_text_col": "text",
    "topic_col": "topic",
    "label_col": "direction",
    "label2id": LABEL2ID,
    "id2label": ID2LABEL,
    "max_length": MAX_LENGTH,
    "topics": TOPICS,
    "class_weight_mode": CLASS_WEIGHT_MODE,
    "class_weight_cap": CLASS_WEIGHT_CAP,
    "metric_for_best_model": "macro_f1",
    "valid_size": VALID_SIZE,
    "random_state": SEED,
    "epochs": EPOCHS,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
}

with open(FINAL_MODEL_DIR / "direction_model_config.json", "w", encoding="utf-8") as file:
    json.dump(direction_model_config, file, ensure_ascii=False, indent=2)

print("Final model directory:", FINAL_MODEL_DIR)
print("Files:")
for path in sorted(FINAL_MODEL_DIR.iterdir()):
    print(" -", path.name)

## 26. Архивирование и скачивание результатов из Kaggle

Kaggle не всегда разрешает автоматическое скачивание файлов на компьютер из кода notebook. Поэтому надёжный способ — ZIP-архивы плюс `FileLink`. Архивы также остаются в `/kaggle/working` и доступны во вкладке Output.

In [ ]:
final_model_zip = make_zip_and_link(
    FINAL_MODEL_DIR,
    f"rubert_{MODEL_SIZE}_direction_binary_{CLASS_WEIGHT_MODE}_final",
)

In [ ]:
reset_dir(ALL_OUTPUTS_PACKAGE_DIR)

important_paths = [
    DATASET_DIR,
    ERROR_DIR,
    FINAL_MODEL_DIR,
    WORK_DIR / "direction_test_predictions.csv",
    WORK_DIR / "direction_metrics.json",
    WORK_DIR / "direction_classification_report.csv",
    WORK_DIR / "direction_confusion_matrix.csv",
    WORK_DIR / "direction_topic_metrics.csv",
    WORK_DIR / "direction_class_weights.csv",
    WORK_DIR / "direction_label_distribution_by_split.csv",
    WORK_DIR / "direction_topic_distribution_train.csv",
    WORK_DIR / "direction_topic_distribution_valid.csv",
    WORK_DIR / "direction_topic_distribution_test.csv",
]

for path in important_paths:
    copy_path_if_exists(path, ALL_OUTPUTS_PACKAGE_DIR)

all_outputs_zip = make_zip_and_link(ALL_OUTPUTS_PACKAGE_DIR, "direction_all_outputs")
display(FileLink("/kaggle/working/direction_all_outputs.zip"))

## 27. Inference helper для финального пайплайна

Эта функция нужна для финального пайплайна: topic model сначала определяет релевантные темы, а затем direction model получает отдельную строку для каждой найденной темы.

In [ ]:
def build_direction_inference_df(posts_df, text_col="text", topic_pred_suffix="_pred"):
    rows = []

    for _, row in posts_df.iterrows():
        for topic in TOPICS:
            pred_col = f"{topic}_relevant{topic_pred_suffix}"

            if pred_col not in posts_df.columns:
                continue

            value = row[pred_col]

            if pd.isna(value):
                continue

            try:
                is_topic_predicted = int(value) == 1
            except (TypeError, ValueError):
                is_topic_predicted = str(value).strip().lower() in {"1", "true", "yes", "y", "да"}

            if is_topic_predicted:
                text = str(row[text_col]).strip()
                rows.append(
                    {
                        "post_uid": row["post_uid"] if "post_uid" in posts_df.columns else None,
                        "text": text,
                        "topic": topic,
                        "model_text": f"[TOPIC={topic}] {text}",
                    }
                )

    return pd.DataFrame(rows)

## 28. Ожидаемые выходные файлы

После полного запуска на Kaggle ожидаются:

```text
/kaggle/working/direction_dataset/direction_train_prepared.parquet
/kaggle/working/direction_dataset/direction_valid_prepared.parquet
/kaggle/working/direction_dataset/direction_test_prepared.parquet

/kaggle/working/direction_label_distribution_by_split.csv
/kaggle/working/direction_topic_distribution_train.csv
/kaggle/working/direction_topic_distribution_valid.csv
/kaggle/working/direction_topic_distribution_test.csv
/kaggle/working/direction_class_weights.csv

/kaggle/working/direction_test_predictions.csv
/kaggle/working/direction_metrics.json
/kaggle/working/direction_classification_report.csv
/kaggle/working/direction_confusion_matrix.csv
/kaggle/working/direction_topic_metrics.csv

/kaggle/working/direction_errors/direction_errors_all.csv
/kaggle/working/direction_errors/direction_error_summary.csv
/kaggle/working/direction_errors/t1_direction_errors.csv
/kaggle/working/direction_errors/t2_direction_errors.csv
/kaggle/working/direction_errors/t3_direction_errors.csv
/kaggle/working/direction_errors/t4_direction_errors.csv
/kaggle/working/direction_errors/t5_direction_errors.csv
/kaggle/working/direction_errors/plus_predicted_as_minus.csv
/kaggle/working/direction_errors/minus_predicted_as_plus.csv

/kaggle/working/rubert_large_direction_binary_balanced_final/

/kaggle/working/direction_prepared_data.zip
/kaggle/working/direction_reports.zip
/kaggle/working/rubert_large_direction_binary_balanced_final.zip
/kaggle/working/direction_all_outputs.zip
```

Основной рекомендуемый запуск:

```python
DATA_INPUT_MODE = "presplit"
MODEL_SIZE = "large"
CLASS_WEIGHT_MODE = "balanced"
MAX_LENGTH = 512
EPOCHS = 4
```